# PGM E0.2 — Local Relational Necessity

Source-locked execution of Issue #78. The immutable E0.1-v533 negative verdict and E0.1b evidence are not modified. This notebook uses official Train supervision and PublicTest development labels only after all unsupervised dictionaries and features are frozen. PrivateTest is forbidden.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Irthn1311/FER2013_Graph.git'
REPO_BRANCH = 'research/pixel-relational-motif-e02'
SOURCE_SHA = 'd980037fe5d004200ffd7cbb2f1b86176529e46a'
EXPECTED_DICTIONARY_SHA256 = '68154a054f712bb07692146904bcba57f10e079c7efc92723aa0bccba9f6273b'
EXPECTED_TRAIN_SHA256 = 'deb82c4b4e01b90776a718c34934666b0bdde6696ca1d0149f8fe807a8ff4ba8'
EXPECTED_PUBLIC_SHA256 = '412036d077c6ec203047b2935ab14bc858d8136ee26e8db3e23023f1fc9dee08'
FER_ROOT = Path('/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split')
TRAIN_CSV = FER_ROOT / 'train.csv'
PUBLIC_CSV = FER_ROOT / 'val.csv'
OUTPUT_DIR = Path('/kaggle/working/outputs/pixel_relational_motif_e02')
PACKAGE_RELATIVE = Path('research/pixel_relational_motif_e0')
RUN_TESTS = True
RUN_E02 = True


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
WORKING = Path('/kaggle/working')
PROJECT = WORKING / 'FER2013_Graph_E02'
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(PROJECT)], check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',SOURCE_SHA], check=True)
head = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'], text=True).strip()
if head != SOURCE_SHA: raise RuntimeError(f'source lock mismatch: {head} != {SOURCE_SHA}')
subprocess.run(['git','-C',str(PROJECT),'diff','--quiet'], check=True)
subprocess.run(['git','-C',str(PROJECT),'diff','--cached','--quiet'], check=True)
PACKAGE = PROJECT / PACKAGE_RELATIVE
PACKAGE_SRC = PACKAGE / 'src'
sys.path.insert(0, str(PACKAGE_SRC))
import pixel_relational_motif_e0
imported = Path(pixel_relational_motif_e0.__file__).resolve()
if PACKAGE_SRC.resolve() not in imported.parents: raise RuntimeError(f'import isolation violation: {imported}')
print('Expected scientific source SHA:', SOURCE_SHA)
print('Actual checked-out source SHA:', head)
print('Source lock PASS')


In [ ]:
if RUN_TESTS:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PACKAGE_SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    result = subprocess.run([sys.executable,'-m','pytest',str(PACKAGE/'tests'),'-q'], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0: raise RuntimeError(f'pre-data pytest failed: {result.returncode}')
    print('Pre-data regression tests PASS')


In [ ]:
import platform, time
import numpy as np, scipy, sklearn
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': np.__version__,
    'scipy': scipy.__version__,
    'sklearn': sklearn.__version__,
    'source_sha': SOURCE_SHA,
}
print('Environment:', json.dumps(environment, indent=2))


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

if not TRAIN_CSV.is_file(): raise FileNotFoundError(TRAIN_CSV)
if not PUBLIC_CSV.is_file(): raise FileNotFoundError(PUBLIC_CSV)
if sha256(TRAIN_CSV) != EXPECTED_TRAIN_SHA256: raise RuntimeError('Train SHA mismatch')
if sha256(PUBLIC_CSV) != EXPECTED_PUBLIC_SHA256: raise RuntimeError('Public SHA mismatch')
candidates = sorted(Path('/kaggle/input').rglob('e01_dictionary.npz'))
matches = [path for path in candidates if sha256(path) == EXPECTED_DICTIONARY_SHA256]
if len(matches) != 1: raise RuntimeError(f'need exactly one frozen v533 dictionary; matches={matches}')
DICTIONARY_NPZ = matches[0]
print('Train SHA256 PASS:', EXPECTED_TRAIN_SHA256)
print('Public SHA256 PASS:', EXPECTED_PUBLIC_SHA256)
print('Frozen v533 dictionary SHA256 PASS:', sha256(DICTIONARY_NPZ))
print('No PrivateTest path is configured or inspected.')


In [ ]:
import contextlib, threading
@contextlib.contextmanager
def heartbeat(label, interval_seconds=180):
    stop = threading.Event(); started = time.time()
    def worker():
        while not stop.wait(interval_seconds): print(f'[heartbeat] {label}: {(time.time()-started)/60:.1f} min', flush=True)
    thread = threading.Thread(target=worker, daemon=True); thread.start()
    try: yield
    finally:
        stop.set(); thread.join(timeout=1); print(f'[heartbeat] {label}: complete {(time.time()-started)/60:.1f} min', flush=True)


In [ ]:
from pixel_relational_motif_e0.e02_runner import run_e02
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2, sort_keys=True), encoding='utf-8')
if RUN_E02:
    with heartbeat('PGM E0.2 scientific execution'):
        summary = run_e02(TRAIN_CSV, PUBLIC_CSV, DICTIONARY_NPZ, OUTPUT_DIR)
    print(json.dumps({
        'formal_verdict': summary['formal_verdict'],
        'metrics': summary['metrics'],
        'paired_bootstrap': summary['paired_bootstrap'],
        'private_test_read': summary['private_test_read'],
    }, indent=2))


In [ ]:
required = {'e02_summary.json','e02_results.npz',*[f'e02_control_seed{seed}.npz' for seed in [42,43,44,45,46]]}
present = {path.name for path in OUTPUT_DIR.iterdir() if path.is_file()}
missing = sorted(required - present)
if missing: raise RuntimeError(f'missing E0.2 artifacts: {missing}')
manifest = {path.name: {'sha256': sha256(path), 'bytes': path.stat().st_size} for path in sorted(OUTPUT_DIR.iterdir()) if path.is_file()}
manifest_path = OUTPUT_DIR / 'e02_artifact_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True, allow_nan=False), encoding='utf-8')
print('Artifact manifest:', json.dumps(manifest, indent=2))
print('E0.2 execution complete. E0.1/E0.1b unchanged; E0.3 and M0 not started.')
